# Step 8: Streaming Inference with Model Monitoring

This notebook demonstrates continuous inference using the **REST endpoint** while the Model Monitor tracks predictions in real-time.

### What This Notebook Does

1. **Generates streaming patient data** - Simulates new patient encounters
2. **Calls REST endpoint** - Makes real-time predictions via SPCS service
3. **Stores predictions** - Writes to MODEL_PREDICTIONS table with features
4. **Enables monitoring** - Model Monitor detects drift by comparing against baseline

### Prerequisites

- Model deployed with REST service (notebook 06)
- Model Monitor created (notebook 07)
- Snowflake PAT token for REST authentication

## Imports and Configuration

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import os
import sys
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from source.configs import get_config
from source.utils import get_session

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

## Streaming Data Simulation

In [ ]:
%autoreload
import importlib
import data.simulator
importlib.reload(data.simulator)
from data.simulator import StreamingDataSimulator

simulator = StreamingDataSimulator(
    session=session,
    database=config.snowflake.database,
    schema_name=config.snowflake.schema_name,
)

print("Simulator initialized")

## Configure REST Endpoint

In [ ]:
import requests
from source.utils import get_feature_config

feature_config = get_feature_config(config)
NUMERIC_FEATURES = feature_config["all_numeric_features"]
CATEGORICAL_FEATURES = feature_config["all_categorical_features"]

registry = Registry(
    session,
    database_name=config.snowflake.database,
    schema_name=config.snowflake.schema_name,
)

model = registry.get_model(config.model.model_name)
versions = model.versions()
MODEL_VERSION = versions[-1]
print(f"Model version to monitor: {MODEL_VERSION}")

rest_endpoint = MODEL_VERSION.list_services().iloc[0].inference_endpoint

if rest_endpoint:
    REST_URL = f"https://{rest_endpoint}/predict"
    print(f"REST Endpoint: {REST_URL}")
else:
    print("ERROR: Could not get REST endpoint. Make sure the service is running.")
    REST_URL = None

## Authentication

In [ ]:
import os

PAT_TOKEN = os.environ.get("SNOWFLAKE_TOKEN", None)
if PAT_TOKEN is None:
  PAT_TOKEN = input("Enter your Snowflake PAT token: ").strip()

headers = {
    "Authorization": f'Snowflake Token="{PAT_TOKEN}"',
    "Content-Type": "application/json",
}

print("Authentication configured")

## Continuous Streaming Inference

Run this cell to start making predictions. With **autocapture enabled**, every inference is automatically logged to `INFERENCE_TABLE`.

Open the **Model Monitoring** tab in Snowsight to watch metrics update in real-time.

In [ ]:
def compute_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute engineered features from raw patient data."""
    logger.info("Computing engineered features")

    result = df.copy()
    result.columns = [c.upper() for c in result.columns]

    result["SHOCK_INDEX"] = np.where(
        result["SYSTOLIC_BP"] > 0, result["HEART_RATE"] / result["SYSTOLIC_BP"], np.nan
    )

    result["PULSE_PRESSURE"] = result["SYSTOLIC_BP"] - result["DIASTOLIC_BP"]

    result["BMI_CATEGORY"] = pd.cut(
        result["BMI"],
        bins=[0, 18.5, 25, 30, 35, 100],
        labels=["UNDERWEIGHT", "NORMAL", "OVERWEIGHT", "OBESE", "SEVERELY_OBESE"],
    ).astype(str)

    result["VITAL_SIGNS_SEVERITY"] = (
        (result["HEART_RATE"] > 100).astype(int)
        + (result["HEART_RATE"] < 60).astype(int)
        + (result["RESPIRATORY_RATE"] > 20).astype(int)
        + (result["OXYGEN_SATURATION"] < 94).astype(int) * 2
        + (result["TEMPERATURE"] > 38.0).astype(int)
        + (result["SYSTOLIC_BP"] < 90).astype(int) * 2
        + (result["SYSTOLIC_BP"] > 180).astype(int)
    )

    logger.info(f"Computed {len(COMPUTED_FEATURES)} engineered features")

    return result

In [ ]:
import time
import pandas as pd
from snowflake.ml.registry import Registry
from source.utils import get_feature_config

feature_config = get_feature_config()
FEATURE_COLUMNS = feature_config['all_numeric_features'] + feature_config['all_categorical_features']
RISK_LEVELS = feature_config['class_labels']
COMPUTED_FEATURES = feature_config['computed_features']

registry = Registry(session, database_name=config.snowflake.database, schema_name=config.snowflake.schema_name)
model = registry.get_model(config.model.model_name)
MODEL_VERSION = model.versions()[-1].version_name

print(f"Using {len(FEATURE_COLUMNS)} features (including {len(COMPUTED_FEATURES)} computed)")

def compute_features_for_records(records: list) -> list:
    """Compute engineered features for streaming records."""
    df = pd.DataFrame(records)
    df.columns = [c.upper() for c in df.columns]
    features_df = compute_engineered_features(df)
    return features_df.to_dict('records')

def predict_batch_rest(records: list) -> list:
    """Call REST endpoint with computed features."""
    records_with_features = compute_features_for_records(records)
    feature_records = [{k: v for k, v in r.items() if k in FEATURE_COLUMNS} for r in records_with_features]
    payload = {"dataframe_records": feature_records}
    
    response = requests.post(REST_URL, headers=headers, json=payload, timeout=30)
    if response.ok:
        result = response.json()
        predictions = result.get("data", result) if isinstance(result, dict) else result
        return predictions
    else:
        print(f"REST Error: {response.status_code} - {response.text}")
        return None

BATCH_SIZE = 10
NUM_BATCHES = 10
INTERVAL_SECONDS = 5

print(f"Starting continuous inference: {NUM_BATCHES} batches of {BATCH_SIZE} records")
print(f"Total predictions: {NUM_BATCHES * BATCH_SIZE}")
print(f"Interval: {INTERVAL_SECONDS} seconds between batches")
print(f"Autocapture: Enabled - predictions logged automatically to INFERENCE_TABLE")
print("-" * 50)

total_predictions = 0
for batch_num in range(NUM_BATCHES):
    records = [simulator.generate_streaming_record() for _ in range(BATCH_SIZE)]
    
    predictions = predict_batch_rest(records)
    if predictions:
        total_predictions += len(predictions)
        print(f"Batch {batch_num + 1}/{NUM_BATCHES}: {len(predictions)} predictions (Total: {total_predictions})")
    else:
        print(f"Batch {batch_num + 1}/{NUM_BATCHES}: FAILED")
    
    if batch_num < NUM_BATCHES - 1:
        time.sleep(INTERVAL_SECONDS)

print("-" * 50)
print(f"Complete! {total_predictions} predictions made")
print("Predictions are automatically captured in INFERENCE_TABLE")

## Verify Auto-Captured Predictions

In [ ]:
MODEL_NAME = config.model.model_name

print("Auto-captured inference logs:")
session.sql(f"""
    SELECT 
        TIMESTAMP,
        RECORD_ATTRIBUTES:"snow.model_serving.request.data.AGE"::NUMBER AS AGE,
        RECORD_ATTRIBUTES:"snow.model_serving.request.data.ADMISSION_TYPE"::VARCHAR AS ADMISSION_TYPE,
        RECORD_ATTRIBUTES:"snow.model_serving.response.data.output_feature_0"::VARCHAR AS PREDICTION,
        RECORD_ATTRIBUTES:"snow.model_serving.response.code"::NUMBER AS STATUS
    FROM TABLE(INFERENCE_TABLE('{MODEL_NAME}'))
    WHERE RECORD_ATTRIBUTES:"snow.model_serving.function.name" = 'predict'
    ORDER BY TIMESTAMP DESC
    LIMIT 10
""").show()

print("\nTotal auto-captured predictions:")
session.sql(f"""
    SELECT COUNT(*) as total_predictions
    FROM TABLE(INFERENCE_TABLE('{MODEL_NAME}'))
    WHERE RECORD_ATTRIBUTES:"snow.model_serving.function.name" = 'predict'
""").show()

# Pipeline Complete!

With **autocapture enabled**, every inference request is automatically logged.

## What's Captured

| Field | Description |
|-------|-------------|
| Request data | All input features (AGE, BMI, etc.) |
| Response data | Model predictions |
| Timestamps | Request/response times |
| Metadata | Model version, service info |

## View Model Monitor

The monitor uses `INFERENCE_LOGS_VIEW` (built on `INFERENCE_TABLE`) as its source.

Open **Snowsight → AI & ML → Models → PATIENT_RISK_MODEL → Monitors** to see:
- Prediction distribution over time
- Feature drift compared to baseline
- Segment-level analysis (by ADMISSION_TYPE, INSURANCE_TYPE)